# GEE tier

Here I am investigating the maximum tier and computational process I can use per-task. This will guide on how to split Guinea in several regions given by their ecoregion.

Once we stablish it, this notebook will also hold the first investigation with Hanses deforestation maps.

## 1. Setup

In [ ]:
import json
import time
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent                     # .../Guinea_Methodology
KEY_FILE = PROJECT_ROOT / "gee-account.json"         # service-account key (git-ignored, never commit it)
ECOREGIONS_GPKG = Path.cwd() / "output_datasets" / "ecoregions_guinea.gpkg"

## 2. Connect to Google Earth Engine

The key file is a Google Cloud **service account** key. Three things are read from it:

| Field | What it is |
|---|---|
| `client_email` | The identity of the service account (who is asking) |
| `project_id` | The Google Cloud project that is billed / whose quota is used |
| `private_key` | The secret used to sign in (never printed) |

In [ ]:
with open(KEY_FILE) as f:
    key_info = json.load(f)

SERVICE_ACCOUNT = key_info["client_email"]
GEE_PROJECT = key_info["project_id"]

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, key_file=str(KEY_FILE))
ee.Initialize(credentials=credentials, project=GEE_PROJECT)

print("Service account:", SERVICE_ACCOUNT)
print("Cloud project:  ", GEE_PROJECT)
print("Test request:   ", ee.String("connected").getInfo())

## 3. What can this account process?

### 3.1 What Earth Engine says about the project

`registrationState` tells whether the project is registered for **commercial** or **noncommercial** use.
Noncommercial projects get a free monthly allowance of compute, measured in **EECU-hours** (Earth Engine Compute
Units). The allowance depends on the project's
[noncommercial tier](https://developers.google.com/earth-engine/guides/noncommercial_tiers); a project that never
picked a tier is on the **Community tier (150 EECU-hours / month)**. Going over does not stop the work, it runs in a
slower "restricted mode". The tier also sets how many batch exports can run at the same time.

The API does not report the tier itself: check it in the Cloud Console under
*Earth Engine → Configuration*, and the usage under *Monitoring* (see
[Earth Engine quotas](https://developers.google.com/earth-engine/guides/usage)).

In [ ]:
project_config = ee.data.getProjectConfig()
project_config

### 3.2 Measuring the cost of a request

Each request can be profiled: Earth Engine reports how many **EECU-seconds** it used. The helper below runs a
request, and returns its result, the wall-clock time and the EECU-seconds, so every test in this notebook can be
compared against the monthly budget (150 EECU-hours = 540,000 EECU-seconds).

In [ ]:
MONTHLY_BUDGET_EECU_SECONDS = 150 * 3600  # Community tier, change if the project is on a higher tier

# Earth Engine stores the profile of each request under an ID; this function fetches profiles by ID.
get_profiles = ee.ApiFunction.lookup("Profile.getProfiles").call


def run_and_measure(ee_object, profile_attempts=5):
    """Compute an Earth Engine object and return (result, seconds, eecu_seconds).

    eecu_seconds is None if the profile could not be fetched (it is sometimes not ready yet on the server).
    """
    profile_ids = []
    start = time.perf_counter()
    with ee.data.profiling(profile_ids.append):
        result = ee_object.getInfo()
    seconds = time.perf_counter() - start

    for attempt in range(profile_attempts):
        try:
            profile_text = get_profiles(ids=profile_ids).getInfo()
            break
        except ee.EEException:
            time.sleep(2 * (attempt + 1))
    else:
        return result, seconds, None

    # The profile is a table; its first column is EECU-seconds per operation ("-" when negligible).
    eecu_seconds = 0.0
    for line in profile_text.splitlines()[1:]:
        columns = line.split()
        if columns and columns[0] != "-":
            eecu_seconds += float(columns[0])
    return result, seconds, round(eecu_seconds, 1)

## 4. Hansen Global Forest Change

[Hansen et al. (2013)](https://doi.org/10.1126/science.1244693), version `2025_v1_13`, from the University of Maryland.

### How Hansen computes forest loss

- **Input**: every Landsat scene (30 m) since 2000. For each pixel and year a set of spectral metrics is computed and a
  machine-learning model (decision trees trained on visually interpreted samples) decides whether the pixel changed.
- **"Tree"** means any vegetation **taller than 5 m**. `treecover2000` is the % canopy closure of that vegetation in the
  year 2000 (0–100).
- **"Loss"** is a *stand-replacing disturbance*: the pixel goes from having tree canopy to having (almost) none.
  It is recorded once, with the year it happened in `lossyear` (1 = 2001 … 25 = 2025, 0 = no loss).
- **No forest definition is applied.** Loss is mapped for any pixel with some tree cover in 2000, so the user has to
  choose a canopy threshold (Global Forest Watch uses **> 30 %**) to decide what counts as forest.
- **It measures tree cover loss, not deforestation.** Hansen does not know *why* trees disappeared: clearing for
  farming, mining or urban growth, but also fire, storms, logging that later regrows, or plantation harvest are all
  "loss". In a city, cutting the trees of a garden or a street to build is loss; a city that stays a city is not.
- The method changed over time: years from 2011 on were reprocessed with an improved model and Landsat 8 is used
  from 2013, so recent years detect much more small-scale loss. **The yearly series is not consistent**, and the
  University of Maryland advises against comparing early and recent years directly.

### Bands

| Band | Meaning |
|---|---|
| `treecover2000` | Canopy closure in 2000, % (0–100) |
| `loss` | 1 = lost tree cover at some point 2001–2025 |
| `lossyear` | Year of loss, 1–25 (2001–2025), 0 = no loss |
| `gain` | 1 = tree cover gain 2000–2012 (not updated after 2012) |
| `first_b30/40/50/70` | Landsat red, NIR, SWIR1, SWIR2 composite, around year 2000 |
| `last_b30/40/50/70` | Same composite for the last year (2025) |
| `datamask` | 0 = no data, 1 = land, 2 = water |

The native grid is EPSG:4326 at 0.00025° (≈ 27.4 × 27.6 m at Guinea's latitude). All counts below use that exact
grid so each pixel is counted once.

In [ ]:
hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")

HANSEN_CRS = "EPSG:4326"
HANSEN_TRANSFORM = [0.00025, 0, -180, 0, -0.00025, 80]  # native 0.00025° grid

print(hansen.bandNames().getInfo())

## 5. Study cell: 1° x 1° around Conakry

Cell `W014N09` of the 1° grid from `ecoregions.ipynb` (lon −14 to −13, lat 9 to 10). It holds Conakry, its suburbs, the
coastal mangroves and farmland inland, so it is a good place to see how Hansen behaves in a city. The cell is cut to
Guinea's border so the sea and Sierra Leone are left out. Change `CELL_WEST` / `CELL_SOUTH` to study another cell.

In [ ]:
CELL_WEST, CELL_SOUTH = -14, 9   # lower-left corner of the 1° cell
CELL_SIZE_DEG = 1

guinea_gdf = gpd.read_file(ECOREGIONS_GPKG, layer="guinea_boundary")
guinea = ee.Geometry(guinea_gdf.geometry.iloc[0].simplify(0.001).__geo_interface__)  # ~100 m simplification

cell_square = ee.Geometry.Rectangle([CELL_WEST, CELL_SOUTH, CELL_WEST + CELL_SIZE_DEG, CELL_SOUTH + CELL_SIZE_DEG])
cell = cell_square.intersection(guinea, maxError=1)

print(f"Cell area inside Guinea: {cell.area(maxError=1).divide(1e6).getInfo():,.0f} km²")

## 6. How many pixels were lost in the cell?

Pixels are counted on Hansen's native grid, only on land (`datamask == 1`). `.unweighted()` counts a pixel as 1 as
soon as its centre is inside the cell (the default would give border pixels a fraction).

Loss is counted for several canopy thresholds, to show how much the answer depends on the forest definition:
threshold 0 = every pixel Hansen flagged, 30 = the usual Global Forest Watch definition.

In [ ]:
CANOPY_THRESHOLDS = [0, 10, 30, 50, 75]

land = hansen.select("datamask").eq(1)
tree_cover_2000 = hansen.select("treecover2000")
loss = hansen.select("loss")

pixel_layers = [ee.Image(1).rename("land_pixels")]
for threshold in CANOPY_THRESHOLDS:
    pixel_layers.append(tree_cover_2000.gt(threshold).rename(f"forest2000_pixels_gt{threshold}"))
    pixel_layers.append(loss.And(tree_cover_2000.gt(threshold)).rename(f"loss_pixels_gt{threshold}"))
pixel_image = ee.Image.cat(pixel_layers).updateMask(land)

pixel_counts_request = pixel_image.reduceRegion(
    reducer=ee.Reducer.sum().unweighted(),
    geometry=cell,
    crs=HANSEN_CRS,
    crsTransform=HANSEN_TRANSFORM,
    maxPixels=1e10,
)
pixel_counts, seconds, eecu = run_and_measure(pixel_counts_request)
print(f"Computed in {seconds:.1f} s, {eecu} EECU-seconds")

pixels_table = pd.DataFrame(
    {
        "canopy_threshold_%": CANOPY_THRESHOLDS,
        "forest_2000_pixels": [pixel_counts[f"forest2000_pixels_gt{t}"] for t in CANOPY_THRESHOLDS],
        "loss_pixels": [pixel_counts[f"loss_pixels_gt{t}"] for t in CANOPY_THRESHOLDS],
    }
)
pixels_table["loss_%_of_forest_2000"] = 100 * pixels_table["loss_pixels"] / pixels_table["forest_2000_pixels"]
print(f"Land pixels in the cell: {pixel_counts['land_pixels']:,.0f}")
pixels_table

## 7. How much area was lost?

Pixels in EPSG:4326 do not all have the same area (they shrink towards the poles), so area is computed by summing
`ee.Image.pixelArea()` (m² of each pixel) over the loss pixels, instead of multiplying the count by a fixed size.

In [ ]:
area_image = pixel_image.multiply(ee.Image.pixelArea()).divide(1e6)  # km² per pixel

area_request = area_image.reduceRegion(
    reducer=ee.Reducer.sum().unweighted(),
    geometry=cell,
    crs=HANSEN_CRS,
    crsTransform=HANSEN_TRANSFORM,
    maxPixels=1e10,
)
areas, seconds, eecu = run_and_measure(area_request)
print(f"Computed in {seconds:.1f} s, {eecu} EECU-seconds")

areas_table = pd.DataFrame(
    {
        "canopy_threshold_%": CANOPY_THRESHOLDS,
        "forest_2000_km2": [areas[f"forest2000_pixels_gt{t}"] for t in CANOPY_THRESHOLDS],
        "loss_2001_2025_km2": [areas[f"loss_pixels_gt{t}"] for t in CANOPY_THRESHOLDS],
    }
)
areas_table["mean_pixel_area_m2"] = 1e6 * areas_table["loss_2001_2025_km2"] / pixels_table["loss_pixels"]
print(f"Land area of the cell: {areas['land_pixels']:,.0f} km²")
areas_table.round(1)

### 7.1 Loss per year

Same area calculation, split by `lossyear`, with the 30 % canopy threshold.

Look for a jump around 2013: when it appears in every cell, it comes from the change of method and Landsat 8, not
from a real change on the ground. Trends should only be read within 2013–2025.

In [ ]:
CANOPY_THRESHOLD = 30

loss_area_by_year_request = (
    ee.Image.pixelArea().divide(1e6)
    .addBands(hansen.select("lossyear"))
    .updateMask(land.And(loss).And(tree_cover_2000.gt(CANOPY_THRESHOLD)))
    .reduceRegion(
        reducer=ee.Reducer.sum().unweighted().group(groupField=1, groupName="lossyear"),
        geometry=cell,
        crs=HANSEN_CRS,
        crsTransform=HANSEN_TRANSFORM,
        maxPixels=1e10,
    )
)
loss_by_year, seconds, eecu = run_and_measure(loss_area_by_year_request)
print(f"Computed in {seconds:.1f} s, {eecu} EECU-seconds")

loss_by_year = pd.DataFrame(loss_by_year["groups"]).rename(columns={"sum": "loss_km2"})
loss_by_year["year"] = 2000 + loss_by_year["lossyear"]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(loss_by_year["year"], loss_by_year["loss_km2"], color="#c0392b")
ax.set_ylabel("Tree cover loss (km²)")
ax.set_title(f"Hansen loss per year, canopy > {CANOPY_THRESHOLD}%, cell {CELL_WEST}, {CELL_SOUTH}")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### 7.2 Where does the loss end up? (land cover today)

To see how much of Hansen's loss is in cities, every loss pixel is looked up in
[ESA WorldCover 2021](https://esa-worldcover.org/) (10 m). WorldCover shows what the land is **after** the loss, so a
loss pixel that is now *Built-up* is a tree that was replaced by a building or road. Loss that is now *Tree cover* is
regrowth (e.g. fallow in shifting cultivation), a temporary disturbance or a false detection.

Only loss from **2001–2020** is used here: WorldCover is a 2021 map, so loss from 2022 onwards would still appear as
*Tree cover* and bias the result. This is computed for both "every flagged pixel" (0 %) and the 30 % threshold.

In [ ]:
LAST_LOSS_YEAR_BEFORE_WORLDCOVER = 20  # lossyear 20 = 2020

WORLDCOVER_CLASSES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland", 50: "Built-up",
    60: "Bare / sparse vegetation", 70: "Snow and ice", 80: "Permanent water", 90: "Herbaceous wetland",
    95: "Mangroves", 100: "Moss and lichen",
}
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().rename("worldcover")


def loss_area_by_worldcover(canopy_threshold):
    request = (
        ee.Image.pixelArea().divide(1e6)
        .addBands(worldcover)
        .updateMask(
            land.And(loss)
            .And(tree_cover_2000.gt(canopy_threshold))
            .And(hansen.select("lossyear").lte(LAST_LOSS_YEAR_BEFORE_WORLDCOVER))
        )
        .reduceRegion(
            reducer=ee.Reducer.sum().unweighted().group(groupField=1, groupName="worldcover"),
            geometry=cell,
            crs=HANSEN_CRS,
            crsTransform=HANSEN_TRANSFORM,  # WorldCover is sampled at each Hansen pixel centre
            maxPixels=1e10,
        )
    )
    result, seconds, eecu = run_and_measure(request)
    print(f"Threshold {canopy_threshold}%: computed in {seconds:.1f} s, {eecu} EECU-seconds")
    table = pd.DataFrame(result["groups"]).set_index("worldcover")["sum"].rename(f"loss_km2_gt{canopy_threshold}")
    return table


loss_by_landcover = pd.concat([loss_area_by_worldcover(0), loss_area_by_worldcover(30)], axis=1).fillna(0)
loss_by_landcover.index = loss_by_landcover.index.map(WORLDCOVER_CLASSES)
for column in list(loss_by_landcover.columns):
    loss_by_landcover[column.replace("km2", "%")] = 100 * loss_by_landcover[column] / loss_by_landcover[column].sum()
loss_by_landcover.sort_values("loss_km2_gt0", ascending=False).round(1)

## 8. Visual check against satellite imagery

Layers, from bottom to top (toggle them in the layer control):

- **Landsat ~2000** and **Landsat 2025**: Hansen's own `first_*` / `last_*` composites, shown as false colour
  (SWIR1 / NIR / red): vegetation is green, bare soil and buildings are pink-purple.
- **Sentinel-2 2025 dry season** (Nov 2024 – Mar 2025, true colour, 10 m): an independent and sharper image. Clouds are
  removed with [Cloud Score+](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_CLOUD_SCORE_PLUS_V1_S2_HARMONIZED)
  and the median of the clear pixels is taken. The dry season is used because Guinea is cloudy in the rainy season.
- **Tree cover 2000**, **Loss year** (yellow = 2001 → red = 2025, 30 % threshold) and **WorldCover built-up**.

The map opens on Conakry. Compare loss pixels with the two dates: if the area was trees in 2000 and is houses or bare
soil in 2025, Hansen is right; if it looks the same, it is a false detection or a temporary disturbance.

In [ ]:
S2_START, S2_END = "2024-11-01", "2025-03-31"
CLEAR_THRESHOLD = 0.60  # Cloud Score+ "cs" ≥ 0.6 keeps clear pixels

cloud_score = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
sentinel2_2025 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(cell_square)
    .filterDate(S2_START, S2_END)
    .linkCollection(cloud_score, ["cs"])
    .map(lambda image: image.updateMask(image.select("cs").gte(CLEAR_THRESHOLD)))
    .median()
    .clip(cell_square)
)
print("Sentinel-2 scenes used:",
      ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(cell_square).filterDate(S2_START, S2_END).size().getInfo())

sentinel2_vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 2500}
landsat_2000_vis = {"bands": ["first_b50", "first_b40", "first_b30"], "min": 0, "max": 150}
landsat_2025_vis = {"bands": ["last_b50", "last_b40", "last_b30"], "min": 0, "max": 150}
tree_cover_vis = {"bands": ["treecover2000"], "min": 0, "max": 100, "palette": ["black", "green"]}
loss_year_vis = {"bands": ["lossyear"], "min": 1, "max": 25, "palette": ["yellow", "orange", "red"]}

loss_forest = hansen.select("lossyear").updateMask(loss.And(tree_cover_2000.gt(CANOPY_THRESHOLD)))
built_up = worldcover.eq(50).selfMask()

m = geemap.Map(center=[9.60, -13.62], zoom=12)
m.add_layer(hansen.clip(cell_square), landsat_2000_vis, "Landsat ~2000 (Hansen first)", False)
m.add_layer(hansen.clip(cell_square), landsat_2025_vis, "Landsat 2025 (Hansen last)", False)
m.add_layer(sentinel2_2025, sentinel2_vis, "Sentinel-2 2025 dry season")
m.add_layer(hansen.clip(cell_square), tree_cover_vis, "Tree cover 2000", False)
m.add_layer(built_up.clip(cell_square), {"palette": ["#00bcd4"]}, "WorldCover 2021 built-up", False)
m.add_layer(loss_forest.clip(cell_square), loss_year_vis, f"Loss year (canopy > {CANOPY_THRESHOLD}%)")
m.add_layer(ee.FeatureCollection([ee.Feature(cell)]).style(color="white", fillColor="00000000", width=2), {}, "Study cell")
m

## 9. How much can the account process in one interactive request?

The same calculation (loss area, 30 % threshold, 30 m native grid) is repeated over larger and larger areas, from a
quarter-degree square up to the whole of Guinea. For each run the wall-clock time and the EECU-seconds are recorded,
plus what share of the monthly budget it used. Interactive requests (`getInfo`) stop after about 5 minutes; if a size
fails here, it has to be sent as a batch export (`ee.batch.Export`) or split into smaller cells.

In [ ]:
loss_area_image = ee.Image.pixelArea().divide(1e6).updateMask(land.And(loss).And(tree_cover_2000.gt(CANOPY_THRESHOLD)))

test_regions = {
    # small squares are anchored on the north-east (inland) corner of the study cell and cut to Guinea
    "0.25° square": ee.Geometry.Rectangle([CELL_WEST + 0.75, CELL_SOUTH + 0.75, CELL_WEST + 1, CELL_SOUTH + 1]).intersection(guinea, 1),
    "0.5° square": ee.Geometry.Rectangle([CELL_WEST + 0.5, CELL_SOUTH + 0.5, CELL_WEST + 1, CELL_SOUTH + 1]).intersection(guinea, 1),
    "1° cell (study cell)": cell,
    "2° square (inland)": ee.Geometry.Rectangle([-11, 9, -9, 11]),
    "Whole of Guinea": guinea,
}

capacity_rows = []
for name, region in test_regions.items():
    request = loss_area_image.reduceRegion(
        reducer=ee.Reducer.sum().unweighted(),
        geometry=region,
        crs=HANSEN_CRS,
        crsTransform=HANSEN_TRANSFORM,
        maxPixels=1e13,
    )
    try:
        result, seconds, eecu = run_and_measure(request)
        status, loss_km2 = "ok", result["area"]
    except ee.EEException as error:
        status, loss_km2, seconds, eecu = f"failed: {error}", None, None, None
    capacity_rows.append(
        {
            "region": name,
            "region_area_km2": region.area(maxError=1).divide(1e6).getInfo(),
            "status": status,
            "loss_km2": loss_km2,
            "seconds": seconds,
            "eecu_seconds": eecu,
        }
    )
    print(f"{name}: {status}")

capacity = pd.DataFrame(capacity_rows)
capacity["approx_pixels_millions"] = capacity["region_area_km2"] * 1e6 / 760 / 1e6  # ~760 m² per Hansen pixel
capacity["%_of_monthly_budget"] = 100 * capacity["eecu_seconds"] / MONTHLY_BUDGET_EECU_SECONDS
capacity.round(3)

## 10. Notes for the methodology

Things to read off the results above before choosing how to compute forest loss for all of Guinea:

- **Yearly series** (section 7.1): a step around 2013 means only 2013–2025 can be used for trends.
- **Canopy threshold** (section 6–7): how much of the loss disappears between 0 % and 30 %? Loss at low canopy is
  mostly savanna trees, farm trees and urban trees, not forest.
- **Land cover after loss** (section 7.2): loss that is now *Built-up* is urban expansion over trees (real tree loss,
  but land-use change to settlement, not forest-to-farm). Loss that is now *Tree cover* is regrowth or noise.
- **Visual check** (section 8): sample loss pixels and check them against Landsat 2000 and Sentinel-2 2025.
- **Processing** (section 9): if the whole country runs interactively, splitting Guinea into cells is only needed for
  heavier processing (e.g. building your own Sentinel-2 composites and classifiers), not for Hansen statistics.